In [6]:
import time

import pytz
from tinkoff.invest import Client, CandleInterval
from tinkoff.invest.utils import now
from datetime import timedelta,datetime,timezone
import numpy as np
import pandas as pd
import datetime
import matplotlib.pyplot as plt
import scipy.stats as stats



In [30]:
TOKEN = "" #Введите token T-Api

## Парсер опционов
# В данном куске кода идёт парсинг опционов через T-API, в том числе:
- parsing_datetime - время парсинга
- code - код опциона
- type - тип опционы(Call/Put)
- strike - страйк опциона
- T - время до эспирации в годах
- bid - минимальная цена, за которую готовы купить опцион
- ask - минимальная цена, за которую готовы продать опцион
- BA - цена базового актива
- mid - значение расчитанное по формуле 1/2(bid + ask)
- spread_pct - значение расчитанное по формуле (ask - mid)/mid
- moneyness - значение обозначающие актив находится в деньгах или нет (BA/strike)
- volume - количество заявок

In [31]:

ISIN = 'RU000A107UL4'

with Client(TOKEN) as client:
    # Получаем инструмент по FIGI
    instrument = client.instruments.find_instrument(query='SBER')
    for i in instrument.instruments:
        if i.api_trade_available_flag == True and i.instrument_type == 'share':
            print(i.api_trade_available_flag)
            print(i.instrument_type)
            print(i.uid)
            print(i.figi)


True
share
e6123145-9665-43e0-8413-cd61b8aa9b13
BBG004730N88
True
share
c190ff1f-1447-4227-b543-316332699ca5
BBG0047315Y7


In [32]:

UID = "e6123145-9665-43e0-8413-cd61b8aa9b13"

with Client(TOKEN) as client:
    response = client.instruments.options_by(basic_asset_uid="3888b26d-19f3-4a0b-a212-f36b82d6212c")




In [33]:

rows = []
def q_to_float(q):
    return q.units + q.nano / 1e9
with Client(TOKEN) as client:
    response = client.instruments.options()





/var/folders/f5/kzrq0kqx07sb2j5ffctwkm4w0000gn/T/ipykernel_8288/2646219249.py:5: DeprecatedWarning: options is deprecated. Use `Client.instruments.options_by(...)` method instead
  response = client.instruments.options()


In [34]:
response.instruments[0]

Option(uid='354736a5-cf7b-4760-935a-dc169fa58b20', position_uid='1fb329e7-51b4-4574-b4f6-639ad191b26f', ticker='VK180CL8', class_code='SPBOPT', basic_asset_position_uid='3888b26d-19f3-4a0b-a212-f36b82d6212c', trading_status=<SecurityTradingStatus.SECURITY_TRADING_STATUS_NORMAL_TRADING: 5>, real_exchange=<RealExchange.REAL_EXCHANGE_MOEX: 1>, direction=<OptionDirection.OPTION_DIRECTION_CALL: 2>, payment_type=<OptionPaymentType.OPTION_PAYMENT_TYPE_PREMIUM: 1>, style=<OptionStyle.OPTION_STYLE_EUROPEAN: 2>, settlement_type=<OptionSettlementType.OPTION_EXECUTION_TYPE_CASH_SETTLEMENT: 2>, name='ВК CALL 180₽ 20.12', currency='rub', settlement_currency='', asset_type='TYPE_SECURITY', basic_asset='VKCO', exchange='FORTS', country_of_risk='RU', country_of_risk_name='Российская Федерация', sector='SECTOR_IT', brand=BrandData(logo_name='vkc.png', logo_base_color='#000000', text_color='#ffffff'), lot=1, basic_asset_size=Quotation(units=1, nano=0), klong=Quotation(units=0, nano=0), kshort=Quotation(u

In [35]:
Options = []
for i in response.instruments:
    if i.basic_asset=="SBER":
        Options.append(i)

In [38]:

with Client(TOKEN) as client:
    for i in Options:
        time.sleep(0.2)
        parsing_time = datetime.datetime.now()
        base_price_resp = client.market_data.get_last_prices(
        instrument_id=[UID]
    )
        base_price = q_to_float(base_price_resp.last_prices[0].price)
        # стакан
        orderbook = client.market_data.get_order_book(
            instrument_id=i.uid,
            depth=1
        )

        bid = q_to_float(orderbook.bids[0].price) if orderbook.bids else None
        ask = q_to_float(orderbook.asks[0].price) if orderbook.asks else None

        # цена контракта (если lot > 1)
        contract_bid = bid * i.lot if bid else None
        contract_ask = ask * i.lot if ask else None
        option_uid = i.uid
        target_date = datetime.datetime(2026, 2, 25)
        time.sleep(0.2)
        candles = list(client.get_all_candles(
        instrument_id=option_uid,
        from_=target_date,
        to=target_date + timedelta(days=1),
        interval=CandleInterval.CANDLE_INTERVAL_DAY))

        rows.append({
            "parsing_datetime": parsing_time,
            "code": i.ticker,
            "type": i.direction.name.replace("OPTION_DIRECTION_", ""),
            "strike": q_to_float(i.strike_price),
            "T": i.expiration_date.date(),
            "bid": bid,
            "ask": ask,
            "bid": contract_bid,
            "ask": contract_ask,
            "BA": base_price,
            "volume": candles[0].volume if candles else 0

        })
df = pd.DataFrame(rows)

In [41]:
df.head()

,parsing_datetime,code,type,strike,T,bid,ask,BA,volume
0,2026-02-25 23:01:23.365000,SR125CI9,CALL,125.0,2029-09-19,135.0,NaN,317.1,0
1,2026-02-25 23:01:23.705650,SR185CU9,PUT,185.0,2029-09-19,NaN,NaN,317.1,0
2,2026-02-25 23:01:23.981718,SR350CI8,CALL,350.0,2028-09-20,NaN,NaN,317.1,0
3,2026-02-25 23:01:24.253901,SR270CR6,PUT,270.0,2026-06-17,NaN,3.0,317.1,0
4,2026-02-25 23:01:24.519739,SR315CX9,PUT,315.0,2029-12-19,NaN,NaN,317.1,0


In [42]:
df.fillna(value=0, inplace=True)
df['mid'] = 0.5*(df['bid'] + df['ask'])
df['spread_pct'] = (df['ask'] - df['bid'])/df['mid']
df['moneyness'] = df['BA']/df['strike']
today = datetime.datetime.now()
df['T'] = pd.to_datetime(df['T'])
df['T'] = (df['T'] - today).dt.days / 365.25

In [43]:
df['T'] = df['T'].round(3)

In [44]:
df.head()

,parsing_datetime,code,type,strike,T,bid,ask,BA,volume,mid,spread_pct,moneyness
0,2026-02-25 23:01:23.365000,SR125CI9,CALL,125.0,3.562,135.0,0.0,317.1,0,67.5,-2.0,2.536800
1,2026-02-25 23:01:23.705650,SR185CU9,PUT,185.0,3.562,0.0,0.0,317.1,0,0.0,NaN,1.714054
2,2026-02-25 23:01:23.981718,SR350CI8,CALL,350.0,2.565,0.0,0.0,317.1,0,0.0,NaN,0.906000
3,2026-02-25 23:01:24.253901,SR270CR6,PUT,270.0,0.304,0.0,3.0,317.1,0,1.5,2.0,1.174444
4,2026-02-25 23:01:24.519739,SR315CX9,PUT,315.0,3.811,0.0,0.0,317.1,0,0.0,NaN,1.006667


In [45]:
df.to_csv('Options_SBER.csv')


## Парсер Акций СберБанка
# В данном куске кода идёт парсинг акций через T-API, в том числе:
- time - дата, на которую есть цена
- r - лог доходность
- volatility20 - волатильность с окном в 20 дней
- volatility60 - волатильность с окном в 60 дней

In [47]:
FIGI = "BBG004730N88"

to_date = now()
from_date = to_date - timedelta(days=365*6)
candles_list = []
with Client(TOKEN) as client:
    for candle in client.get_all_candles(
        figi=FIGI,
        from_=from_date,
        to=to_date,
        interval=CandleInterval.CANDLE_INTERVAL_DAY,
    ):
        candles_list.append({
            "time": candle.time,
            "close": candle.close.units + candle.close.nano / 1e9,
        })


df = pd.DataFrame(candles_list)
df.reset_index(drop=True, inplace=True)
df.sort_values(by="time", inplace=True)

print(df.tail())

                          time   close
1572 2026-02-21 00:00:00+00:00  313.79
1573 2026-02-22 00:00:00+00:00  313.54
1574 2026-02-23 00:00:00+00:00  314.16
1575 2026-02-24 00:00:00+00:00  316.15
1576 2026-02-25 00:00:00+00:00  317.30


In [48]:
df['r'] = np.log(df['close'] / df['close'].shift(1))
df['volatility20'] = df['r'].rolling(window=20).std()
df['volatility60'] = df['r'].rolling(window=60).std()

In [49]:
df.head()

,time,close,r,volatility20,volatility60
0,2020-02-27 00:00:00+00:00,242.88,NaN,NaN,NaN
1,2020-02-28 00:00:00+00:00,233.36,-0.039985,NaN,NaN
2,2020-03-02 00:00:00+00:00,228.17,-0.022491,NaN,NaN
3,2020-03-03 00:00:00+00:00,236.63,0.036407,NaN,NaN
4,2020-03-04 00:00:00+00:00,235.27,-0.005764,NaN,NaN


In [5]:
df.to_csv('Sber.csv')

NameError: name 'df' is not defined

## Загрузка данных


In [7]:
Sber = pd.read_csv('SBER.csv')
Options_SBER = pd.read_csv('Options_SBER.csv')

In [41]:
class Black_Sholse:
    def __init__(self,S_0,K,option_type,T,volatility,r = np.log(1+0.1429)):
        self.S_0 = S_0
        self.K = K
        self.option_type = option_type
        self.T = T
        self.r = r
        self.volatility = volatility
    def d1_d2_counter(self):
        d1 = ((np.log(self.S_0) - np.log(self.K)) + (self.r + 1/2 * self.volatility**2) * self.T)/ (np.sqrt(self.volatility**2 * self.T))
        d2 = d1 - (np.sqrt(self.volatility**2 * self.T))
        self.d1 = d1
        self.d2 = d2
    def Black_Sholse_call(self):
        self.d1_d2_counter()
        call = self.S_0 * stats.norm.cdf(self.d1) - self.K * stats.norm.cdf(self.d2) * np.exp(-self.r * self.T)
        return call
    def Black_Sholse_put(self):
        self.d1_d2_counter()
        put = self.K * stats.norm.cdf(-self.d2) * np.exp(-self.r * self.T) - self.S_0 * stats.norm.cdf(-self.d1)
        return put
    def Black_Sholse(self):
        self.greeks()
        if self.option_type == 'CALL':
            return self.Black_Sholse_call()
        else:
            return self.Black_Sholse_put()
    def greeks(self):
        self.d1_d2_counter()
        d1 = self.d1
        d2 = self.d2

        pdf = stats.norm.pdf(d1)

        delta_call = stats.norm.cdf(d1)
        delta_put = delta_call - 1

        gamma = pdf / (self.S_0 * self.volatility * np.sqrt(self.T))

        vega = self.S_0 * np.sqrt(self.T) * pdf

        theta_call = (-self.S_0 * pdf * self.volatility / (2 * np.sqrt(self.T))
                      - self.r * self.K * np.exp(-self.r * self.T) * stats.norm.cdf(d2))

        theta_put = (-self.S_0 * pdf * self.volatility / (2 * np.sqrt(self.T))
                     + self.r * self.K * np.exp(-self.r * self.T) * stats.norm.cdf(-d2))

        rho_call = self.K * self.T * np.exp(-self.r * self.T) * stats.norm.cdf(d2)
        rho_put = -self.K * self.T * np.exp(-self.r * self.T) * stats.norm.cdf(-d2)
        if self.option_type == 'CALL':
            self.delta = delta_call
            self.theta = theta_call
            self.vega = vega
            self.rho = rho_call
            self.gama = gamma
        else:
            self.gamma = gamma
            self.delta = delta_put
            self.theta = theta_put
            self.vega = vega
            self.rho = rho_put



# c d

In [ ]:
print(1)

# Delta hedge

In [9]:
Options_SBER.head()

,Unnamed: 0,parsing_datetime,code,type,strike,T,bid,ask,BA,volume,mid,spread_pct,moneyness
0,0,2026-02-25 23:01:23.365000,SR125CI9,CALL,125.0,3.562,135.0,0.0,317.1,0,67.5,-2.0,2.536800
1,1,2026-02-25 23:01:23.705650,SR185CU9,PUT,185.0,3.562,0.0,0.0,317.1,0,0.0,NaN,1.714054
2,2,2026-02-25 23:01:23.981718,SR350CI8,CALL,350.0,2.565,0.0,0.0,317.1,0,0.0,NaN,0.906000
3,3,2026-02-25 23:01:24.253901,SR270CR6,PUT,270.0,0.304,0.0,3.0,317.1,0,1.5,2.0,1.174444
4,4,2026-02-25 23:01:24.519739,SR315CX9,PUT,315.0,3.811,0.0,0.0,317.1,0,0.0,NaN,1.006667


In [11]:
Sber.tail()

,Unnamed: 0,time,close,r,volatility20,volatility60
1572,1572,2026-02-21 00:00:00+00:00,313.79,-0.000669,0.005010,0.005487
1573,1573,2026-02-22 00:00:00+00:00,313.54,-0.000797,0.005027,0.005485
1574,1574,2026-02-23 00:00:00+00:00,314.16,0.001975,0.005025,0.005483
1575,1575,2026-02-24 00:00:00+00:00,316.15,0.006314,0.004721,0.005531
1576,1576,2026-02-25 00:00:00+00:00,317.30,0.003631,0.004677,0.005508


In [21]:
opt = Options_SBER[
    (Options_SBER['moneyness'] > 0.95) &
    (Options_SBER['moneyness'] < 1.25)
]

In [48]:
opt

,Unnamed: 0,parsing_datetime,code,type,strike,T,bid,ask,BA,volume,mid,spread_pct,moneyness
3,3,2026-02-25 23:01:24.253901,SR270CR6,PUT,270.0,0.304,0.00,3.00,317.10,0,1.500,2.000000,1.174444
4,4,2026-02-25 23:01:24.519739,SR315CX9,PUT,315.0,3.811,0.00,0.00,317.10,0,0.000,NaN,1.006667
6,6,2026-02-25 23:01:25.050479,SR260CB6D,CALL,260.0,-0.003,0.00,0.00,317.10,0,0.000,NaN,1.219615
13,13,2026-02-25 23:01:26.888993,SR290CX8,PUT,290.0,2.815,0.00,0.00,317.10,0,0.000,NaN,1.093448
16,16,2026-02-25 23:01:27.658754,SR270CX8,PUT,270.0,2.815,0.00,0.00,317.10,0,0.000,NaN,1.174444
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1985,1985,2026-02-25 23:16:21.500818,SR330CC6,CALL,330.0,0.055,1.45,2.06,317.23,5556,1.755,0.347578,0.961303
1988,1988,2026-02-25 23:16:22.943231,SR265CX8,PUT,265.0,2.815,0.00,0.00,317.14,0,0.000,NaN,1.196755
1992,1992,2026-02-25 23:16:24.853115,SR305CX8,PUT,305.0,2.815,0.00,0.00,317.14,0,0.000,NaN,1.039803
1997,1997,2026-02-25 23:16:27.206400,SR330CO6A,PUT,330.0,0.016,11.51,13.25,317.29,4009,12.380,0.140549,0.961485


In [49]:
option = opt.loc[1985,:]
option

Unnamed: 0                                1985
parsing_datetime    2026-02-25 23:16:21.500818
code                                  SR330CC6
type                                      CALL
strike                                   330.0
T                                        0.055
bid                                       1.45
ask                                       2.06
BA                                      317.23
volume                                    5556
mid                                      1.755
spread_pct                            0.347578
moneyness                             0.961303
Name: 1985, dtype: object

In [50]:

share = Sber[Sber['time'] == '2026-02-25 00:00:00+00:00']
share

,Unnamed: 0,time,close,r,volatility20,volatility60
1576,1576,2026-02-25 00:00:00+00:00,317.3,0.003631,0.004677,0.005508


In [51]:
BS = Black_Sholse(K=option['strike'],
                  option_type=option['type'],
                  T=option['T'],
                  volatility=share['volatility20'] *  np.sqrt(252),
                  S_0=share['close'],
)
BS.Black_Sholse()

1576    0.073938
Name: close, dtype: float64

In [55]:
print(BS.delta)
'''
продаём 0.03414072 акции
'''

[0.03414072]


'\nпродаём 0.03414072 акции\n'

# Стоимость стратегии

In [57]:
# Сделки по mid
cost = option['mid'] - 0.03414072*share['close']
print(cost)
# покупка по ask продажа по bid
cost = option['ask'] - 0.03414072*share['close']
print(cost)


1576   -9.07785
Name: close, dtype: float64
1576   -8.77285
Name: close, dtype: float64


# Collar